# 1. Configuración general

In [0]:
import mlflow
import pandas as pd
import numpy as np

from pyspark.sql import functions as F

# ============================================================
# CONFIGURACIÓN DE UNITY CATALOG
# ============================================================

CATALOGO = "workspace"
ESQUEMA = "oceanwatch"

# Tabla de entrada

TABLA_GOLD_ML = (
    f"{CATALOGO}.{ESQUEMA}.ais_gold_ml_features"
)

# Tabla de salida

TABLA_PREDICCIONES = (
    f"{CATALOGO}.{ESQUEMA}.ais_gold_ml_predicciones"
)

# ============================================================
# CONFIGURACIÓN DEL MODELO
# ============================================================

NOMBRE_MODELO = "ais_isolation_forest"

MODELO_REGISTRADO = (
    f"{CATALOGO}.{ESQUEMA}.{NOMBRE_MODELO}"
)

VERSION_MODELO = "1"

URI_MODELO = (
    f"models:/{MODELO_REGISTRADO}/{VERSION_MODELO}"
)

mlflow.set_registry_uri("databricks-uc")

# ============================================================
# CONFIGURACIÓN DEL CHECKPOINT
# ============================================================

RUTA_CHECKPOINT = (
    "/Volumes/workspace/oceanwatch/ais_landing/"
    "control/checkpoints/ml_inferencia_v1/"
)

# ============================================================
# VARIABLES DEL MODELO
# ============================================================

VARIABLES_MODELO = [
    "SOG",
    "Length",
    "Width",
    "LAT",
    "LON",
    "COG",
    "Heading",
    "VesselType"
]

VARIABLES_IDENTIFICACION = [
    "MMSI",
    "BaseDateTime"
]

VARIABLES_TRAZABILIDAD = [
    "fecha",
    "fecha_ingesta",
    "archivo_origen"
]

# ============================================================
# REPORTE DE CONFIGURACIÓN
# ============================================================

print("CONFIGURACIÓN DE INFERENCIA OCEANWATCH")
print("=" * 80)

print(f"Tabla de entrada: {TABLA_GOLD_ML}")
print(f"Tabla de salida: {TABLA_PREDICCIONES}")
print(f"Modelo registrado: {MODELO_REGISTRADO}")
print(f"Versión del modelo: {VERSION_MODELO}")
print(f"URI del modelo: {URI_MODELO}")
print(f"Checkpoint: {RUTA_CHECKPOINT}")

print("\nVARIABLES")
print("=" * 80)

print(f"Variables del modelo: {len(VARIABLES_MODELO)}")
print(f"Variables de identificación: {len(VARIABLES_IDENTIFICACION)}")
print(f"Variables de trazabilidad: {len(VARIABLES_TRAZABILIDAD)}")

print("\nDISPONIBILIDAD DE TABLAS")
print("=" * 80)

print(
    "Tabla de entrada disponible:",
    spark.catalog.tableExists(TABLA_GOLD_ML)
)

print(
    "Tabla de predicciones existente:",
    spark.catalog.tableExists(TABLA_PREDICCIONES)
)

CONFIGURACIÓN DE INFERENCIA OCEANWATCH
Tabla de entrada: workspace.oceanwatch.ais_gold_ml_features
Tabla de salida: workspace.oceanwatch.ais_gold_ml_predicciones
Modelo registrado: workspace.oceanwatch.ais_isolation_forest
Versión del modelo: 1
URI del modelo: models:/workspace.oceanwatch.ais_isolation_forest/1
Checkpoint: /Volumes/workspace/oceanwatch/ais_landing/control/checkpoints/ml_inferencia_v1/

VARIABLES
Variables del modelo: 8
Variables de identificación: 2
Variables de trazabilidad: 3

DISPONIBILIDAD DE TABLAS


{"ts": "2026-11-27 18:00:13.977", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTY2NDI1OTE4MjI0MjE1MxABIAEyJDAxYTIxZTJhLTIzODItN2E5MC1hMjNiLWI5ZjFiZDk2MWFiNjokYzgxY2UxYmItM2NlNi0zMDAzLWE3NjEtYWM1NGI2MThkYWY2SgwIp4qn2AYQgL2eggNQAVgBYAFoxayd9/nEow2IAQA=.", "context": {}}
{"ts": "2026-11-27 18:00:13.977", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTY2NDI1OTE4MjI0MjE1MxABIAEyJDAxYTIxZTJhLTIzODItN2E5MC1hMjNiLWI5ZjFiZDk2MWFiNjokYzgxY2UxYmItM2NlNi0zMDAzLWE3NjEtYWM1NGI2MThkYWY2SgwIp4qn2AYQgL2eggNQAVgBYAFoxayd9/nEow2IAQA=.", "context": {}}
{"ts": "2026-11-27 18:00:13.977", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTY2NDI1OTE4MjI0MjE1MxABIAEyJDAxYTIxZTJhLTIzODItN2E5MC1hMjNiLWI5ZjFiZDk2MWFiNjokYzgxY2UxYmItM2NlNi0zMDAzLWE3NjEtYWM1NGI2

Tabla de entrada disponible: True
Tabla de predicciones existente: True


# 2. Carga del modelo registrado

In [0]:
import mlflow.pyfunc
import time

# ============================================================
# CARGA DEL MODELO
# ============================================================

inicio_carga = time.perf_counter()

modelo_registrado = mlflow.pyfunc.load_model(
    URI_MODELO
)

tiempo_carga = time.perf_counter() - inicio_carga

# ============================================================
# METADATOS DEL MODELO
# ============================================================

firma_modelo = modelo_registrado.metadata.signature

# ============================================================
# REPORTE
# ============================================================

print("CARGA DEL MODELO REGISTRADO")
print("=" * 80)

print(f"Modelo: {MODELO_REGISTRADO}")
print(f"Versión: {VERSION_MODELO}")
print(f"Tiempo de carga: {tiempo_carga:.4f} segundos")

print("\nFIRMA DE ENTRADA")
print("=" * 80)

print(firma_modelo.inputs)

print("\nFIRMA DE SALIDA")
print("=" * 80)

print(firma_modelo.outputs)

print("\n[OK] Modelo cargado correctamente.")

CARGA DEL MODELO REGISTRADO
Modelo: workspace.oceanwatch.ais_isolation_forest
Versión: 1
Tiempo de carga: 8.9385 segundos

FIRMA DE ENTRADA
['SOG': double (required), 'Length': double (required), 'Width': double (required), 'LAT': double (required), 'LON': double (required), 'COG': double (required), 'Heading': integer (required), 'VesselType': integer (required)]

FIRMA DE SALIDA
['Anomaly_Score': double (required), 'Umbral_global': double (required), 'Umbral_categoria': double (optional), 'Categoria_conocida': boolean (required), 'Umbral_categoria_disponible': boolean (required), 'Anomalia_global': boolean (required), 'Anomalia_categoria': boolean (optional), 'Codigo_clasificacion': long (required), 'Clasificacion_final': string (required)]

[OK] Modelo cargado correctamente.


# 3. Definición del esquema de salida

In [0]:
# ============================================================
# BLOQUE 3 - DEFINICIÓN DEL ESQUEMA DE SALIDA
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    DoubleType,
    BooleanType,
    LongType,
    StringType,
    TimestampType
)

# ============================================================
# ESQUEMA ORIGINAL DE GOLD ML FEATURES
# ============================================================

esquema_entrada = spark.table(TABLA_GOLD_ML).schema

# ============================================================
# CAMPOS GENERADOS POR EL MODELO
# ============================================================

campos_prediccion = [

    StructField("Anomaly_Score", DoubleType(), False),

    StructField("Umbral_global", DoubleType(), False),

    StructField("Umbral_categoria", DoubleType(), True),

    StructField("Categoria_conocida", BooleanType(), False),

    StructField("Umbral_categoria_disponible", BooleanType(), False),

    StructField("Anomalia_global", BooleanType(), False),

    StructField("Anomalia_categoria", BooleanType(), True),

    StructField("Codigo_clasificacion", LongType(), False),

    StructField("Clasificacion_final", StringType(), False)

]

# ============================================================
# ESQUEMA DE INFERENCIA DISTRIBUIDA
# ============================================================

esquema_inferencia = StructType(
    esquema_entrada.fields + campos_prediccion
)

# ============================================================
# CAMPOS DE TRAZABILIDAD DE INFERENCIA
# ============================================================

campos_trazabilidad_ml = [

    StructField("Version_modelo", StringType(), False),

    StructField("Fecha_inferencia", TimestampType(), False)

]

# ============================================================
# ESQUEMA FINAL DE LA TABLA DE PREDICCIONES
# ============================================================

esquema_predicciones = StructType(
    esquema_inferencia.fields + campos_trazabilidad_ml
)

# ============================================================
# REPORTE
# ============================================================

print("ESQUEMA DE GOLD ML PREDICCIONES")
print("=" * 80)

print(f"Campos originales: {len(esquema_entrada.fields)}")
print(f"Campos de predicción: {len(campos_prediccion)}")
print(f"Campos de trazabilidad ML: {len(campos_trazabilidad_ml)}")
print(f"Total de campos: {len(esquema_predicciones.fields)}")

print("\nESTRUCTURA DE CAMPOS")
print("=" * 80)

for campo in esquema_predicciones.fields:

    print(
        f"{campo.name:35} "
        f"{campo.dataType.simpleString():15} "
        f"Nullable: {campo.nullable}"
    )

ESQUEMA DE GOLD ML PREDICCIONES
Campos originales: 13
Campos de predicción: 9
Campos de trazabilidad ML: 2
Total de campos: 24

ESTRUCTURA DE CAMPOS
MMSI                                string          Nullable: True
BaseDateTime                        timestamp       Nullable: True
SOG                                 double          Nullable: True
COG                                 double          Nullable: True
Heading                             int             Nullable: True
VesselType                          int             Nullable: True
Length                              double          Nullable: True
Width                               double          Nullable: True
LAT                                 double          Nullable: True
LON                                 double          Nullable: True
fecha                               date            Nullable: True
fecha_ingesta                       timestamp       Nullable: True
archivo_origen                      string     

# 4. Funcion de inferencia

In [0]:
# ============================================================
# BLOQUE 4 - FUNCIÓN DE INFERENCIA DISTRIBUIDA
# ============================================================

import cloudpickle
import pandas as pd

from typing import Iterator

# ============================================================
# SERIALIZACIÓN DEL MODELO REGISTRADO
# ============================================================

modelo_serializado = cloudpickle.dumps(
    modelo_registrado
)

variables_inferencia = list(VARIABLES_MODELO)

columnas_salida = esquema_inferencia.fieldNames()

# ============================================================
# FUNCIÓN DE INFERENCIA DISTRIBUIDA
# ============================================================

def inferencia_distribuida(
    iterator: Iterator[pd.DataFrame]
) -> Iterator[pd.DataFrame]:

    modelo_local = cloudpickle.loads(
        modelo_serializado
    )

    for pdf in iterator:

        pdf = pdf.reset_index(drop=True)

        X_lote = pdf[variables_inferencia].copy()

        predicciones = modelo_local.predict(X_lote)

        predicciones = predicciones.reset_index(drop=True)

        resultado = pd.concat(
            [pdf, predicciones],
            axis=1
        )

        yield resultado[columnas_salida]

# ============================================================
# REPORTE
# ============================================================

print("CONFIGURACIÓN DE INFERENCIA DISTRIBUIDA")
print("=" * 80)

print(
    f"Tamaño del modelo serializado: "
    f"{len(modelo_serializado) / 1024**2:.2f} MB"
)

print("[OK] Función de inferencia configurada.")

CONFIGURACIÓN DE INFERENCIA DISTRIBUIDA
Tamaño del modelo serializado: 1.35 MB
[OK] Función de inferencia configurada.


# 5. Procesamiento Incremental y almacenamiento

In [0]:
import time

from pyspark.sql import functions as F

# ============================================================
# LECTURA INCREMENTAL DE GOLD ML FEATURES
# ============================================================

df_stream_ml = (
    spark.readStream
    .format("delta")
    .table(TABLA_GOLD_ML)
)

# ============================================================
# INFERENCIA DISTRIBUIDA
# ============================================================

df_stream_predicciones = (

    df_stream_ml

    .mapInPandas(
        inferencia_distribuida,
        schema=esquema_inferencia
    )

    .withColumn(
        "Version_modelo",
        F.lit(VERSION_MODELO)
    )

    .withColumn(
        "Fecha_inferencia",
        F.current_timestamp()
    )
)

# ============================================================
# ALMACENAMIENTO INCREMENTAL EN DELTA
# ============================================================

inicio_procesamiento = time.perf_counter()

print("INICIANDO PROCESAMIENTO INCREMENTAL")
print("=" * 80)

query_inferencia = (

    df_stream_predicciones

    .writeStream

    .format("delta")

    .outputMode("append")

    .option(
        "checkpointLocation",
        RUTA_CHECKPOINT
    )

    .partitionBy("fecha")

    .trigger(availableNow=True)

    .toTable(TABLA_PREDICCIONES)
)

# Esperar la finalización del procesamiento

query_inferencia.awaitTermination()

tiempo_procesamiento = (
    time.perf_counter() - inicio_procesamiento
)

# ============================================================
# REPORTE DE EJECUCIÓN
# ============================================================

print("\nPROCESAMIENTO INCREMENTAL FINALIZADO")
print("=" * 80)

print(f"Tabla origen: {TABLA_GOLD_ML}")
print(f"Tabla destino: {TABLA_PREDICCIONES}")
print(f"Versión del modelo: {VERSION_MODELO}")

print(
    f"Tiempo de procesamiento: "
    f"{tiempo_procesamiento:.2f} segundos"
)

print(
    "Tabla de predicciones disponible:",
    spark.catalog.tableExists(TABLA_PREDICCIONES)
)

print("\n[OK] Procesamiento finalizado.")

INICIANDO PROCESAMIENTO INCREMENTAL

PROCESAMIENTO INCREMENTAL FINALIZADO
Tabla origen: workspace.oceanwatch.ais_gold_ml_features
Tabla destino: workspace.oceanwatch.ais_gold_ml_predicciones
Versión del modelo: 1
Tiempo de procesamiento: 2.68 segundos
Tabla de predicciones disponible: True

[OK] Procesamiento finalizado.
